# Tracklet Velocity: Benchmark v3 vs Sorcha cases on MJD 61642

Reproduces the scatter + quiver analysis from `tracklet_comparison_sorcha_vs_benchmark.ipynb`
using:
- **Benchmark v3** — proportional population caps (MBA 96.5%, NEO 1.9%, Trojan 1.25%, TNO 0.34%)
  + epoch MJD **61642** (2027-08-25), the busiest single night in every Sorcha case.
- **Sorcha case1 / case2 / case3** — all filtered to night MJD 61642.

**Matching:** each Sorcha ObjID is looked up in the S3M files to get its V-band (e, H);
those are matched against benchmark v3's (e, H) via nearest-neighbour with a tight
tolerance of 1 × 10⁻⁴ in (e, H) space.  Only unique, unambiguous pairs are kept.

**Cross-case comparison (Section 5):** 663 NEO ObjIDs appear on MJD 61642 in all three
Sorcha cases — exactly the same objects, three different linking configurations.  This lets
us ask whether SSP linking changes the *measured* (vλ, vβ) for the same NEO.

In [ ]:
import glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy.spatial import cKDTree

NIGHT   = 61642
MATCH_TOL = 1e-4          # (e, H) distance threshold for a confident match
POP_COLORS = {'NEO': 'tab:red', 'MBA': 'tab:blue', 'TNO': 'gold',
               'Trojan': 'mediumseagreen', 'other': 'grey'}
CASE_LS  = {'case1': '-',  'case2': '--', 'case3': ':'}
CASE_LAB = {
    'case1': 'case1 (tk=1, sep=0.5")',
    'case2': 'case2 (tk=3, sep=0.5" — real LSST)',
    'case3': 'case3 (tk=1, sep≈0, baseline)',
}

In [ ]:
# ── parse S3M files → (ObjID, e, H_V) lookup ─────────────────────────────────
# col index: 0=ObjID, 3=e, 8=H  (same as original tracklet_comparison notebook)
records = []
for f in sorted(glob.glob('S*.s3m')):
    with open(f) as fh:
        for line in fh:
            if line.startswith('!'): continue
            p = line.split()
            if len(p) < 9: continue
            try: records.append((p[0], float(p[3]), float(p[8])))
            except ValueError: pass
s3m_eh = (pd.DataFrame(records, columns=['ObjID','e_s3m','H_s3m'])
          .drop_duplicates('ObjID').set_index('ObjID'))
print(f'S3M lookup: {len(s3m_eh):,} objects')

In [ ]:
# ── load benchmark v3 (mag_bin_label filtered = VDP-scored subset) ─────────────
bv3_raw = pd.read_parquet('docs/benchmark_comparison_s3m_v3.parquet')
bv3 = bv3_raw[bv3_raw.mag_bin_label.notna()].reset_index(drop=True).copy()
bv3['absdlon'] = bv3.dlon_from_antisun_deg.abs()
tree = cKDTree(bv3[['e','H']].values)
print(f'Benchmark v3 (scored): {len(bv3):,} rows')
print(f'  pops: {dict(bv3.population.value_counts())}')

# ── load Sorcha cases, filter to busiest night ────────────────────────────────
cases = {}
for c in ['case1','case2','case3']:
    df = pd.read_parquet(f'docs/sorcha_comparison_{c}.parquet')
    night = df[df.night == NIGHT].copy()
    night = night.join(s3m_eh, on='ObjID').dropna(subset=['e_s3m','H_s3m'])
    cases[c] = night
    print(f'{c} night {NIGHT}: {len(night)} tracklets — {dict(night.population.value_counts())}')

In [ ]:
# ── match each case → benchmark v3 ───────────────────────────────────────────
matched = {}
for c, night in cases.items():
    dist, idx = tree.query(night[['e_s3m','H_s3m']].values, k=1)
    m = dist < MATCH_TOL
    hit = night[m].copy()
    hit_idx = idx[m]
    hit['vlam_b']  = bv3.vlam.values[hit_idx]
    hit['vbeta_b'] = bv3.vbeta.values[hit_idx]
    hit['pop_b']   = bv3.population.values[hit_idx]
    hit['dv']      = np.hypot(hit.vlam - hit.vlam_b, hit.vbeta - hit.vbeta_b)
    matched[c] = hit
    print(f'{c}: {len(hit)} matched pairs  |  pops: {dict(hit.population.value_counts())}')
    print(f'  |Δv| median={hit.dv.median():.3f}  mean={hit.dv.mean():.3f} deg/day')

## Section 1 — Night 61642 Population Summary

In [ ]:
rows = []
for c, night in cases.items():
    vc = night.population.value_counts()
    row = {'Case': c, 'Total tracklets': len(night)}
    for p in ['NEO','MBA','TNO','Trojan','other']:
        n = int(vc.get(p,0))
        row[p] = f'{n} ({100*n/len(night):.1f}%)'
    row['Matched to v3'] = len(matched[c])
    rows.append(row)
print(f'Night MJD {NIGHT} — population breakdown')
display(pd.DataFrame(rows).set_index('Case'))

## Section 2 — Scatter Plot: Benchmark v3 vs Sorcha (all matched pairs)

Reproduces the first image.  Blue triangles = benchmark v3, red circles = Sorcha.  
Gray lines connect the same object across the two pipelines.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharex=True, sharey=True)

for ax, (c, hit) in zip(axes, matched.items()):
    # gray connector lines
    for _, row in hit.iterrows():
        ax.plot([row.vlam_b, row.vlam], [row.vbeta_b, row.vbeta],
                color='gray', lw=0.5, alpha=0.4, zorder=1)

    # benchmark points (triangles)
    ax.scatter(hit.vlam_b, hit.vbeta_b, marker='^', s=30, alpha=0.7,
               color='tab:blue', label=f'Benchmark v3 (n={len(hit)})', zorder=3)

    # sorcha points (circles), coloured by population
    for pop, grp in hit.groupby('population'):
        ax.scatter(grp.vlam, grp.vbeta, marker='o', s=25, alpha=0.75,
                   color=POP_COLORS.get(pop,'grey'), label=f'Sorcha {pop} (n={len(grp)})', zorder=4)

    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(r'$v_\lambda$ (deg/day)')
    ax.legend(fontsize=7, loc='upper right')
    ax.set_title(f'{CASE_LAB[c]}\n{len(hit)} matched pairs — MJD {NIGHT}', fontsize=9)
    ax.grid(alpha=0.2)

axes[0].set_ylabel(r'$v_\beta$ (deg/day)')
fig.suptitle(f'Ecliptic rate space — Benchmark v3 vs Sorcha (MJD {NIGHT})', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/tracklet_scatter_v3_cases.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 3 — Quiver Plot: Velocity Vectors from Origin

Reproduces the second image.  Each arrow = one matched pair.  
Blue = benchmark v3, red = Sorcha.  Pairs are layered so mismatches are visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharex=True, sharey=True)

for ax, (c, hit) in zip(axes, matched.items()):
    zeros = np.zeros(len(hit))
    ax.quiver(zeros, zeros, hit.vlam_b.values, hit.vbeta_b.values,
              color='tab:blue', alpha=0.45, angles='xy', scale_units='xy', scale=1,
              width=0.004, label='Benchmark v3')
    ax.quiver(zeros, zeros, hit.vlam.values,   hit.vbeta.values,
              color='tab:red',  alpha=0.45, angles='xy', scale_units='xy', scale=1,
              width=0.004, label='Sorcha')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(r'$v_\lambda$ (deg/day)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title(f'{CASE_LAB[c]}\n{len(hit)} pairs  |Δv| med={hit.dv.median():.2f} deg/day', fontsize=9)
    ax.grid(alpha=0.2)

axes[0].set_ylabel(r'$v_\beta$ (deg/day)')
fig.suptitle(f'Velocity vectors from origin — Benchmark v3 vs Sorcha (MJD {NIGHT})', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/tracklet_quiver_v3_cases.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — NEO-Only Focus

Zoom in on the NEO matched pairs — the scientifically most important population.  
Scatter + quiver side by side for each case.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10), sharex=True, sharey=True)

for col, (c, hit) in enumerate(matched.items()):
    neo = hit[hit.population == 'NEO']
    n   = len(neo)

    # top row: scatter
    ax = axes[0, col]
    for _, row in neo.iterrows():
        ax.plot([row.vlam_b, row.vlam], [row.vbeta_b, row.vbeta],
                color='gray', lw=0.8, alpha=0.5, zorder=1)
    ax.scatter(neo.vlam_b, neo.vbeta_b, marker='^', s=55, alpha=0.85,
               color='tab:blue', label=f'Benchmark v3 (n={n})', zorder=3)
    ax.scatter(neo.vlam,   neo.vbeta,   marker='o', s=45, alpha=0.85,
               color='tab:red',  label=f'Sorcha (n={n})',        zorder=4)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f'{CASE_LAB[c]} — NEOs only', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25)

    # bottom row: quiver
    ax = axes[1, col]
    zeros = np.zeros(n)
    ax.quiver(zeros, zeros, neo.vlam_b.values, neo.vbeta_b.values,
              color='tab:blue', alpha=0.6, angles='xy', scale_units='xy', scale=1,
              width=0.006, label='Benchmark v3')
    ax.quiver(zeros, zeros, neo.vlam.values,   neo.vbeta.values,
              color='tab:red',  alpha=0.6, angles='xy', scale_units='xy', scale=1,
              width=0.006, label='Sorcha')
    dv_neo = neo.dv
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(r'$v_\lambda$ (deg/day)')
    ax.legend(fontsize=7)
    ax.set_title(f'|Δv| med={dv_neo.median():.2f}  mean={dv_neo.mean():.2f} deg/day', fontsize=8)
    ax.grid(alpha=0.25)

axes[0,0].set_ylabel(r'Scatter — $v_\beta$ (deg/day)')
axes[1,0].set_ylabel(r'Quiver — $v_\beta$ (deg/day)')
fig.suptitle(f'NEO tracklets only — Benchmark v3 vs Sorcha cases (MJD {NIGHT})', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/tracklet_neo_v3_cases.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── |Δv| stats table for NEOs ────────────────────────────────────────────────
rows = []
for c, hit in matched.items():
    neo = hit[hit.population == 'NEO']
    rows.append({
        'Case': c,
        'N_NEO pairs': len(neo),
        '|Δv| median': round(neo.dv.median(), 3),
        '|Δv| mean':   round(neo.dv.mean(),   3),
        '|Δv| p90':    round(neo.dv.quantile(0.9), 3),
        '|Δv| max':    round(neo.dv.max(), 3),
    })
print('=== NEO velocity offset: Sorcha vs Benchmark v3 ===')
display(pd.DataFrame(rows).set_index('Case'))

## Section 5 — Cross-Case Velocity Comparison (same 663 NEOs)

663 NEO ObjIDs appear on MJD 61642 in **all three** Sorcha cases.  
This is a pure same-object, different-linking-configuration test:  
does the SSP linking choice change the measured (vλ, vβ) for the same NEO?

In [ ]:
# ── build common-NEO table ────────────────────────────────────────────────────
nights_neo = {}
for c in ['case1','case2','case3']:
    df = pd.read_parquet(f'docs/sorcha_comparison_{c}.parquet',
                         columns=['ObjID','night','population','vlam','vbeta',
                                  'P_NEO_vdp','P_NEO_d2','dt_min','mean_mag'])
    nights_neo[c] = df[(df.night == NIGHT) & (df.population == 'NEO')].set_index('ObjID')

common_ids = set(nights_neo['case1'].index) & set(nights_neo['case2'].index) & set(nights_neo['case3'].index)
print(f'NEOs common to all 3 cases on night {NIGHT}: {len(common_ids)}')

# align
ids = sorted(common_ids)
c1n = nights_neo['case1'].loc[ids]
c2n = nights_neo['case2'].loc[ids]
c3n = nights_neo['case3'].loc[ids]

dv12 = np.hypot(c1n.vlam - c2n.vlam, c1n.vbeta - c2n.vbeta)
dv13 = np.hypot(c1n.vlam - c3n.vlam, c1n.vbeta - c3n.vbeta)
dv23 = np.hypot(c2n.vlam - c3n.vlam, c2n.vbeta - c3n.vbeta)
print(f'|Δv| case1 vs case2: median={dv12.median():.4f}  mean={dv12.mean():.4f} deg/day')
print(f'|Δv| case1 vs case3: median={dv13.median():.4f}  mean={dv13.mean():.4f} deg/day')
print(f'|Δv| case2 vs case3: median={dv23.median():.4f}  mean={dv23.mean():.4f} deg/day')

In [ ]:
# ── cross-case quiver: case1 (blue) vs case2 (orange) vs case3 (green) ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# left: quiver overlay (all from origin)
ax = axes[0]
zeros = np.zeros(len(ids))
colors = {'case1': 'tab:blue', 'case2': 'tab:orange', 'case3': 'tab:green'}
for c, nn in [('case3', c3n), ('case1', c1n), ('case2', c2n)]:
    ax.quiver(zeros, zeros, nn.vlam.values, nn.vbeta.values,
              color=colors[c], alpha=0.4, angles='xy', scale_units='xy', scale=1,
              width=0.004, label=CASE_LAB[c])
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel(r'$v_\lambda$ (deg/day)'); ax.set_ylabel(r'$v_\beta$ (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_title(f'Same 663 NEOs — velocity quiver overlay')

# right: case1 vs case2 scatter (direct comparison)
ax = axes[1]
ax.scatter(c1n.vlam, c2n.vlam, s=12, alpha=0.5, color='tab:blue', label=r'$v_\lambda$')
ax.scatter(c1n.vbeta, c2n.vbeta, s=12, alpha=0.5, color='tab:red', label=r'$v_\beta$')
lim = max(abs(c1n.vlam).max(), abs(c2n.vlam).max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='1:1')
ax.set_xlabel('case1 velocity component (deg/day)')
ax.set_ylabel('case2 velocity component (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_title('case1 vs case2 velocity components (same NEO, same night)\n'
             f'|Δv| median={dv12.median():.4f} deg/day')

fig.suptitle(f'Cross-case NEO velocity comparison — MJD {NIGHT} (663 common NEOs)', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/tracklet_crosscase_neo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── P_NEO_vdp and P_NEO_d2 consistency across cases for common NEOs ───────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, score_col, title in [
    (axes[0], 'P_NEO_vdp', 'VDP score'),
    (axes[1], 'P_NEO_d2',  'digest2 score'),
]:
    ax.scatter(nights_neo['case1'].loc[ids, score_col],
               nights_neo['case2'].loc[ids, score_col],
               s=12, alpha=0.5, color='tab:blue', label='case1 vs case2')
    ax.scatter(nights_neo['case1'].loc[ids, score_col],
               nights_neo['case3'].loc[ids, score_col],
               s=12, alpha=0.4, color='tab:orange', marker='s', label='case1 vs case3')
    ax.plot([0,1],[0,1],'k--',lw=0.8, label='1:1')
    ax.set_xlabel(f'case1 {title}'); ax.set_ylabel(f'caseX {title}')
    ax.legend(fontsize=8); ax.grid(alpha=0.2)
    ax.set_title(f'{title} consistency across cases\n(663 common NEOs, night {NIGHT})')

plt.tight_layout()
plt.savefig('Figures/tracklet_score_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 6 — Velocity Offset from Benchmark (case1 only)

For each matched pair, compute the residual displacement:
(Δvλ, Δvβ) = (vlam_sorcha − vlam_bench, vbeta_sorcha − vbeta_bench).

The origin (0, 0) = perfect agreement between Sorcha and benchmark.

**Left panel — arrow version:** every matched pair drawn as a quiver arrow from (0,0)
to its offset.  A thick black arrow marks the median offset across all pairs.  Shows
direction and magnitude of the disagreement for each object.

**Right panel — population scatter version:** same offset points plotted as dots,
coloured by population (NEO=red, MBA=blue, TNO=gold, Trojan=green).  The median
is marked with a large ×.  Shows whether different orbital populations cluster at
different offsets — e.g. do NEOs have a systematic direction bias compared to MBAs?

In [ ]:
hit1 = matched['case1'].copy()

# ── remove false NEO matches (median + 3×MAD on NEO |Δv|) before plotting ────
neo_mask = hit1.population == 'NEO'
med_dv   = hit1.loc[neo_mask, 'dv'].median()
mad_dv   = (hit1.loc[neo_mask, 'dv'] - med_dv).abs().median()
thresh   = med_dv + 3 * mad_dv
false_match = neo_mask & (hit1.dv > thresh)
n_dropped = false_match.sum()
hit1_clean = hit1[~false_match].copy()

dvlam  = hit1_clean.vlam  - hit1_clean.vlam_b
dvbeta = hit1_clean.vbeta - hit1_clean.vbeta_b
med_dvlam  = dvlam.median()
med_dvbeta = dvbeta.median()

print(f'Dropped {n_dropped} false NEO matches (|Δv| > {thresh:.3f} deg/day)')
print(f'Plotting {len(hit1_clean)} clean pairs  |  pops: {dict(hit1_clean.population.value_counts())}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── left: arrow version ────────────────────────────────────────────────────────
ax = axes[0]
zeros = np.zeros(len(hit1_clean))
ax.quiver(zeros, zeros, dvlam.values, dvbeta.values,
          color='steelblue', alpha=0.35, angles='xy', scale_units='xy', scale=1,
          width=0.003, label=f'Individual offsets (n={len(hit1_clean)})')
ax.quiver(0, 0, med_dvlam, med_dvbeta,
          color='black', alpha=1.0, angles='xy', scale_units='xy', scale=1,
          width=0.010, headwidth=5, headlength=5,
          label=f'Median offset\n(Δvλ={med_dvlam:.3f}, Δvβ={med_dvbeta:.3f})')
ax.scatter(0, 0, s=80, color='black', zorder=5)
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel(r'$\Delta v_\lambda$ = $v_{\lambda,\rm Sorcha} - v_{\lambda,\rm bench}$ (deg/day)')
ax.set_ylabel(r'$\Delta v_\beta$ = $v_{\beta,\rm Sorcha} - v_{\beta,\rm bench}$ (deg/day)')
ax.legend(fontsize=9, loc='upper right')
ax.set_title('Velocity offset arrows — case1 vs benchmark v3\n'
             f'(0,0) = perfect agreement  [{n_dropped} false NEO matches removed]', fontsize=10)
ax.grid(alpha=0.2)

# ── right: population scatter version ─────────────────────────────────────────
ax = axes[1]
for pop, grp in hit1_clean.groupby('population'):
    idx = grp.index
    ax.scatter(dvlam.loc[idx], dvbeta.loc[idx],
               s=30, alpha=0.7, color=POP_COLORS.get(pop, 'grey'),
               label=f'{pop} (n={len(grp)})', zorder=3)
for pop, grp in hit1_clean.groupby('population'):
    idx = grp.index
    pm_l = dvlam.loc[idx].median()
    pm_b = dvbeta.loc[idx].median()
    ax.scatter(pm_l, pm_b, s=180, marker='x', linewidths=2.5,
               color=POP_COLORS.get(pop, 'grey'), zorder=5)
ax.scatter(med_dvlam, med_dvbeta, s=350, marker='x', linewidths=3,
           color='black', zorder=6, label=f'Overall median\n({med_dvlam:.3f}, {med_dvbeta:.3f})')
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel(r'$\Delta v_\lambda$ (deg/day)')
ax.set_ylabel(r'$\Delta v_\beta$ (deg/day)')
ax.legend(fontsize=8, loc='upper right')
ax.set_title('Velocity offsets by population — case1 vs benchmark v3\n'
             'small × = per-population median,  large black × = overall median', fontsize=10)
ax.grid(alpha=0.2)

fig.suptitle(f'Sorcha − Benchmark velocity residuals  |  case1, MJD {NIGHT}  '
             f'[{n_dropped} false NEO matches removed]', fontsize=12)
plt.tight_layout()
plt.savefig('Figures/tracklet_offset_case1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nOverall median offset: Δvλ={med_dvlam:.4f}  Δvβ={med_dvbeta:.4f} deg/day')
for pop, grp in hit1_clean.groupby('population'):
    idx = grp.index
    print(f'{pop:8s}  n={len(grp):4d}  '
          f'Δvλ med={dvlam.loc[idx].median():.3f}  '
          f'Δvβ med={dvbeta.loc[idx].median():.3f}  '
          f'|Δv| med={grp.dv.median():.3f}')

### Section 6b — NEO-only residuals: confident vs suspicious matches

The right panel above shows NEO offsets up to 2.1 deg/day — far too large to be a real
velocity difference between two pipelines measuring the same object.  These are almost
certainly **false cKDTree matches**: two different NEOs that happen to share similar (e, H).

Threshold: flag a pair as suspicious if its |Δv| exceeds **median + 3×MAD** of the NEO
|Δv| distribution (a robust outlier criterion).  Confident matches are shown in teal,
suspicious in red.  A second panel cross-plots vlam_bench vs vlam_sorcha for confident
pairs only — if matching is clean these should fall on the 1:1 line.

In [ ]:
neo1 = hit1[hit1.population == 'NEO'].copy()
neo1['dvlam']  = neo1.vlam  - neo1.vlam_b
neo1['dvbeta'] = neo1.vbeta - neo1.vbeta_b

# robust outlier threshold: median + 3×MAD
med_dv  = neo1.dv.median()
mad_dv  = (neo1.dv - med_dv).abs().median()
thresh  = med_dv + 3 * mad_dv
neo1['suspicious'] = neo1.dv > thresh

conf = neo1[~neo1.suspicious]
susp = neo1[ neo1.suspicious]
print(f'NEO pairs — total: {len(neo1)},  confident: {len(conf)},  suspicious: {len(susp)}')
print(f'Threshold |Δv| > {thresh:.3f} deg/day  (median={med_dv:.3f}, MAD={mad_dv:.3f})')
print()
if len(susp):
    print('Suspicious pairs (likely false matches):')
    display(susp[['ObjID','vlam','vbeta','vlam_b','vbeta_b','dvlam','dvbeta','dv']].sort_values('dv', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# ── left: full NEO scatter coloured by confidence ─────────────────────────────
ax = axes[0]
ax.scatter(conf.dvlam, conf.dvbeta, s=55, alpha=0.85, color='teal',
           edgecolors='white', linewidths=0.4, label=f'Confident (n={len(conf)})', zorder=3)
ax.scatter(susp.dvlam, susp.dvbeta, s=80, alpha=0.85, color='tab:red', marker='X',
           edgecolors='darkred', linewidths=0.5, label=f'Suspicious |Δv|>{thresh:.2f} (n={len(susp)})', zorder=4)
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
# draw threshold circle
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(thresh*np.cos(theta), thresh*np.sin(theta), 'k--', lw=1, alpha=0.4, label=f'|Δv|={thresh:.2f} circle')
ax.set_xlabel(r'$\Delta v_\lambda$ (deg/day)'); ax.set_ylabel(r'$\Delta v_\beta$ (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_title(f'All NEO offsets — case1 vs benchmark v3\n(n={len(neo1)} total)', fontsize=10)
ax.set_aspect('equal')

# ── middle: confident pairs only, zoomed ─────────────────────────────────────
ax = axes[1]
ax.scatter(conf.dvlam, conf.dvbeta, s=60, alpha=0.9, color='teal',
           edgecolors='white', linewidths=0.4, zorder=3)
# annotate median
med_cl = conf.dvlam.median(); med_cb = conf.dvbeta.median()
ax.scatter(med_cl, med_cb, s=250, marker='x', linewidths=3, color='black', zorder=5,
           label=f'Median ({med_cl:.3f}, {med_cb:.3f})')
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel(r'$\Delta v_\lambda$ (deg/day)'); ax.set_ylabel(r'$\Delta v_\beta$ (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_title(f'Confident NEO pairs only (n={len(conf)})\nZoomed residuals', fontsize=10)
ax.set_aspect('equal')

# ── right: 1:1 check — vlam_bench vs vlam_sorcha for confident pairs ──────────
ax = axes[2]
ax.scatter(conf.vlam_b, conf.vlam, s=55, alpha=0.85, color='teal',
           edgecolors='white', linewidths=0.4, label=r'$v_\lambda$', zorder=3)
ax.scatter(conf.vbeta_b, conf.vbeta, s=55, alpha=0.85, color='darkorange',
           edgecolors='white', linewidths=0.4, marker='s', label=r'$v_\beta$', zorder=3)
lims = [min(conf.vlam_b.min(), conf.vlam.min(), conf.vbeta_b.min(), conf.vbeta.min()) * 1.1,
        max(conf.vlam_b.max(), conf.vlam.max(), conf.vbeta_b.max(), conf.vbeta.max()) * 1.1]
ax.plot(lims, lims, 'k--', lw=1, label='1:1')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Benchmark v3 velocity (deg/day)')
ax.set_ylabel('Sorcha case1 velocity (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2); ax.set_aspect('equal')
ax.set_title(f'1:1 check — confident NEO pairs (n={len(conf)})\n'
             r'Points on dashed line = perfect agreement', fontsize=10)

fig.suptitle(f'NEO residuals: confident vs suspicious matches — case1, MJD {NIGHT}', fontsize=12)
plt.tight_layout()
plt.savefig('Figures/tracklet_neo_residuals_case1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nConfident NEO pairs (|Δv| ≤ {thresh:.3f}):')
print(f'  Δvλ median={med_cl:.4f}  Δvβ median={med_cb:.4f}  |Δv| median={conf.dv.median():.4f} deg/day')

## Section 7 — Completeness–contamination in the 20–21 mag band (VDP vs digest2)

Restrict everything to the single magnitude bin **`mag20`** (mean_mag 20.00–21.00) on
night MJD 61642, and compare **all objects** in that band (not the matched pairs) —
benchmark v3 vs each Sorcha case.  This is the band where both pipelines have a real
NEO sample *and* the population isn't yet gutted by Sorcha's faint-end 5σ cut.

**Curves (house style):** binary NEO vs non-NEO, scored by `P_NEO_vdp` and `P_NEO_d2`.
x-axis = **NEO completeness** (recall %), y-axis = **contamination** (1 − precision, %).
The marker on each curve is the F1-optimal operating point.  AUC (base-rate independent)
is also tabulated so the four panels stay comparable despite very different NEO fractions.

**Read the base rates first (they differ ~35×):**

| dataset | rows | NEO | NEO fraction |
|---|---:|---:|---:|
| benchmark v3 | 6,263 | 25 | 0.4% (full-sky) |
| case1 | 416 | 61 | 14.7% (footprint) |
| case2 | 432 | 60 | 13.9% |
| case3 | 386 | 60 | 15.5% |

Benchmark v3 propagates the whole sky to the epoch, so its 20–21 band is 98.6% MBA;
the Sorcha cases only contain what the survey footprint delivered that night, which is
far NEO-richer.  Contamination is prevalence-sensitive — at 0.4% NEO even a good
classifier shows high contamination — so compare curves within a panel, and use AUC
to compare across panels.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_auc_score

BAND = 'mag20'   # mean_mag 20.00–21.00
COLORS = {'VDP': 'tab:blue', 'digest2': 'tab:orange'}

def roc_xy(yy, s):
    """Precision-recall → (completeness%, contamination%, bestF1, best_compl%, best_contam%)"""
    p, r, _ = precision_recall_curve(yy, s)
    f1 = np.divide(2*p*r, p+r, out=np.zeros_like(p), where=(p+r) > 0)
    bi = int(np.argmax(f1[:-1]))
    return r*100, (1-p)*100, f1[bi], r[bi]*100, (1-p[bi])*100

# ── assemble the four band-restricted datasets (ALL objects, not matched pairs) ─
band = {}
bb = bv3_raw[bv3_raw.mag_bin_label == BAND].copy()
bb['is_neo'] = (bb.population == 'NEO').astype(int)
band['benchmark v3'] = bb
for c in ['case1','case2','case3']:
    df = pd.read_parquet(f'docs/sorcha_comparison_{c}.parquet',
                         columns=['night','population','mag_bin_label','P_NEO_vdp','P_NEO_d2'])
    n = df[(df.night == NIGHT) & (df.mag_bin_label == BAND)].copy()
    n['is_neo'] = (n.population == 'NEO').astype(int)
    band[c] = n

# ── 1×4 completeness–contamination panels ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6), sharex=True, sharey=True)
summary = []
for ax, (name, d) in zip(axes, band.items()):
    yy = d.is_neo.values
    npos = int(yy.sum())
    cv, kv, fv, mcv, mkv = roc_xy(yy, d['P_NEO_vdp'].values)
    cd, kd, fd, mcd, mkd = roc_xy(yy, d['P_NEO_d2'].values)
    auc_v = roc_auc_score(yy, d['P_NEO_vdp'].values)
    auc_d = roc_auc_score(yy, d['P_NEO_d2'].values)

    ax.plot(cv, kv, lw=2, color=COLORS['VDP'],     label=f'VDP  F1={fv:.3f}')
    ax.plot(cd, kd, lw=2, color=COLORS['digest2'], ls='--', label=f'digest2  F1={fd:.3f}')
    ax.scatter([mcv], [mkv], color=COLORS['VDP'],     s=60, zorder=5)
    ax.scatter([mcd], [mkd], color=COLORS['digest2'], s=60, marker='s', zorder=5)
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.set_xlabel('NEO completeness (%)')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title(f'{name}\n{BAND} band — {len(d)} obj, {npos} NEO', fontsize=9)

    summary.append({'dataset': name, 'N': len(d), 'NEO': npos,
                    'VDP F1': round(fv,3), 'VDP compl%': round(mcv,1), 'VDP contam%': round(mkv,1), 'VDP AUC': round(auc_v,3),
                    'd2 F1': round(fd,3),  'd2 compl%': round(mcd,1),  'd2 contam%': round(mkd,1),  'd2 AUC': round(auc_d,3)})

axes[0].set_ylabel('Contamination (%)')
fig.suptitle(f'Completeness–contamination: VDP vs digest2 in the 20–21 mag band  |  MJD {NIGHT}', fontsize=12)
plt.tight_layout()
plt.savefig('Figures/roc_mag20_v3_cases.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== 20–21 mag band (mag20) — completeness/contamination at best-F1, MJD 61642 ===')
display(pd.DataFrame(summary).set_index('dataset'))

### What the 20–21 band curves show

At the F1-optimal operating point (completeness / contamination), and by AUC:

| dataset | VDP compl / contam | digest2 compl / contam | VDP AUC | digest2 AUC |
|---|---|---|---:|---:|
| benchmark v3 | 48% / 61% | 64% / 20% | 0.735 | **0.980** |
| case1 | 61% / 2.6% | 71% / 2.3% | 0.788 | **0.906** |
| case2 | 62% / 7.5% | 70% / 2.3% | 0.788 | **0.908** |
| case3 | 60% / 2.7% | 72% / 0.0% | 0.795 | **0.911** |

**digest2 wins in this band, in every dataset** — higher completeness at equal-or-lower
contamination, and higher AUC throughout.  Two takeaways:

1. **Restricting to one magnitude band removes VDP's edge.**  VDP's strength is separating
   populations that live in different regions of *rate × magnitude* space (fast faint NEOs
   vs slow bright MBAs vs distant TNOs).  Inside a single narrow mag slice that leverage is
   gone — everyone has the same brightness, so only the velocity signal remains, and digest2's
   orbit-based scoring reads that more cleanly here.  The Sorcha cases hold at ~60–62% VDP
   completeness vs ~70–72% for digest2 at comparable contamination.

2. **The three Sorcha cases are essentially identical** (VDP AUC 0.788/0.788/0.795,
   digest2 0.906/0.908/0.911).  Linking configuration does not change per-band separability —
   consistent with Section 5: linking changes *which* objects survive, not how well they're
   classified once present.

The benchmark panel is the odd one out for contamination: at 0.4% NEO prevalence even
digest2 sits at 20% contamination and VDP at 61%, because a handful of MBA false positives
overwhelm the 25 real NEOs.  That's why the benchmark curves must be read *within* the panel
(VDP vs digest2) and cross-panel comparison should lean on AUC, which is prevalence-free.

## Section 8 — Midpoint-centered agreement plot (case1)

A pipeline-agreement diagnostic.  For each matched object we have two points in ecliptic
rate space: benchmark (blue) and Sorcha (red).  Here we subtract each pair's **own midpoint**
M = ½·(benchmark + Sorcha) from both of its points, so every pair is re-centered on (0,0).

After centering, the benchmark point sits at −½·Δv and the Sorcha point at +½·Δv, mirror-
symmetric about the origin.  What survives is *only* the disagreement between pipelines:

- **Both pipelines agree** → every blue and red point collapses into a tight clump at (0,0).
- **Systematic bias** → the two colours separate into opposite lobes (e.g. all red pushed
  one way, all blue the other).
- **Random measurement noise** → a symmetric spread of both colours around the origin.

Both markers are kept visible (per advisor's request) so you can see *which* pipeline sits
on which side of each disagreement, not just the magnitude of it.  case1, 6 false NEO
matches removed.  Gray connectors pass through the origin by construction.

In [ ]:
# ── case1, drop the 6 false NEO matches (same criterion as Section 6) ─────────
h = matched['case1'].copy()
neo_mask = h.population == 'NEO'
_med = h.loc[neo_mask, 'dv'].median()
_mad = (h.loc[neo_mask, 'dv'] - _med).abs().median()
_thr = _med + 3*_mad
h = h[~(neo_mask & (h.dv > _thr))].copy()

# ── per-pair midpoint (= mean of the two points), then center both on it ──────
mid_lam  = 0.5 * (h.vlam_b  + h.vlam)
mid_beta = 0.5 * (h.vbeta_b + h.vbeta)
# benchmark relative to its own midpoint  → lands at −½Δv
bl_lam,  bl_beta  = h.vlam_b - mid_lam, h.vbeta_b - mid_beta
# sorcha relative to its own midpoint     → lands at +½Δv
sl_lam,  sl_beta  = h.vlam   - mid_lam, h.vbeta   - mid_beta

# spread metric: RMS distance of centered points from origin
rms = np.sqrt(np.mean(bl_lam**2 + bl_beta**2))
print(f'case1 midpoint-centered: {len(h)} pairs')
print(f'  RMS spread from origin: {rms:.4f} deg/day  (= ½ · RMS|Δv|)')
print(f'  half-|Δv| median: {(0.5*h.dv).median():.4f}  max: {(0.5*h.dv).max():.4f} deg/day')

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), sharex=True, sharey=True)

# ── left: coloured by PIPELINE (blue=benchmark, red=Sorcha) ───────────────────
ax = axes[0]
for i in range(len(h)):
    ax.plot([bl_lam.iloc[i], sl_lam.iloc[i]], [bl_beta.iloc[i], sl_beta.iloc[i]],
            color='gray', lw=0.4, alpha=0.35, zorder=1)
ax.scatter(bl_lam, bl_beta, s=22, alpha=0.7, color='tab:blue',
           label=f'Benchmark v3 (n={len(h)})', zorder=3)
ax.scatter(sl_lam, sl_beta, s=22, alpha=0.7, color='tab:red',
           label=f'Sorcha case1 (n={len(h)})', zorder=3)
ax.scatter(0, 0, s=90, color='black', marker='+', linewidths=2, zorder=5)
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel(r'$v_\lambda$ − midpoint (deg/day)')
ax.set_ylabel(r'$v_\beta$ − midpoint (deg/day)')
ax.legend(fontsize=9, loc='upper right')
ax.set_title('Centered on per-pair midpoint — coloured by pipeline\n'
             'tight clump at 0 = pipelines agree', fontsize=10)
ax.grid(alpha=0.2); ax.set_aspect('equal')

# ── right: coloured by POPULATION (both markers, benchmark=triangle, sorcha=circle)
ax = axes[1]
for i in range(len(h)):
    ax.plot([bl_lam.iloc[i], sl_lam.iloc[i]], [bl_beta.iloc[i], sl_beta.iloc[i]],
            color='gray', lw=0.4, alpha=0.3, zorder=1)
for pop, grp in h.groupby('population'):
    ii = [h.index.get_loc(x) for x in grp.index]
    col = POP_COLORS.get(pop, 'grey')
    ax.scatter(bl_lam.iloc[ii], bl_beta.iloc[ii], s=30, alpha=0.75, color=col,
               marker='^', edgecolors='white', linewidths=0.3, zorder=3)
    ax.scatter(sl_lam.iloc[ii], sl_beta.iloc[ii], s=28, alpha=0.75, color=col,
               marker='o', edgecolors='white', linewidths=0.3, zorder=3,
               label=f'{pop} (n={len(grp)})')
ax.scatter(0, 0, s=90, color='black', marker='+', linewidths=2, zorder=5)
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel(r'$v_\lambda$ − midpoint (deg/day)')
ax.legend(fontsize=8, loc='upper right', title='△ benchmark  ○ sorcha')
ax.set_title('Centered on per-pair midpoint — coloured by population\n'
             '△ = benchmark, ○ = Sorcha', fontsize=10)
ax.grid(alpha=0.2); ax.set_aspect('equal')

fig.suptitle(f'Midpoint-centered rate-space agreement — case1 vs benchmark v3, MJD {NIGHT}',
             fontsize=12)
plt.tight_layout()
plt.savefig('Figures/tracklet_midpoint_centered_case1.png', dpi=150, bbox_inches='tight')
plt.show()

### Section 8b — Speed-normalized agreement (and why it must be read with care)

Dividing the offset by each object's own speed |v_mid| turns the axes into *fractional*
velocity disagreement.  But this normalization is **ill-conditioned for slow objects**: on
MJD 61642 the MBAs sit near opposition, where a main-belt asteroid's apparent sky rate passes
through its stationary point and |v| → 0.  Dividing a finite offset by a near-zero speed
sends the fraction to absurd values (MBA median ≈ 150%, tail into the thousands of %).

So the fractional metric is only meaningful for genuinely fast movers.  And there the result
is clean and reassuring:

- **NEO median fractional disagreement ≈ 6%** — the fast, science-critical population is
  measured consistently by both pipelines to within a few percent of its own speed.
- **MBA "disagreement" is a denominator artifact** — large fraction because |v| ≈ 0, not
  because the absolute offset is large.  For MBAs the *absolute* plot (Section 8) is the
  honest diagnostic; the fractional one is not.

Left panel proves it's a denominator effect: fractional offset vs midpoint speed (log–log) —
the blowup happens only at low |v|, and the fast NEOs sit low.  Right panel: the fractional
distribution restricted to objects with |v| > 0.3 deg/day, where the metric is well-defined.

In [ ]:
# reuse h, mid_lam, mid_beta from Section 8 (case1, false NEO matches already removed)
speed_mid = np.hypot(mid_lam, mid_beta)          # each object's own speed at the midpoint
frac_dv   = h.dv / speed_mid                       # |Δv| / |v_mid|

print(f'case1 speed-normalized: {len(h)} pairs')
print(f'  fractional |Δv|/|v| — median={frac_dv.median()*100:.1f}%  '
      f'p90={frac_dv.quantile(0.9)*100:.1f}%  max={frac_dv.max()*100:.0f}%')
for pop, grp in h.groupby('population'):
    sp = np.hypot(0.5*(grp.vlam_b+grp.vlam), 0.5*(grp.vbeta_b+grp.vbeta))
    fd = grp.dv / sp
    print(f'  {pop:6s} n={len(grp):4d}  median frac={fd.median()*100:5.1f}%   '
          f'median |v|={sp.median():.3f} deg/day')

FAST = 0.3   # deg/day — threshold above which |v| normalization is well-conditioned

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── left: fractional offset vs midpoint speed (log–log) → proves denominator effect
ax = axes[0]
for pop, grp in h.groupby('population'):
    sp = np.hypot(0.5*(grp.vlam_b+grp.vlam), 0.5*(grp.vbeta_b+grp.vbeta))
    fd = grp.dv / sp
    ax.scatter(sp, fd*100, s=28, alpha=0.7, color=POP_COLORS.get(pop, 'grey'),
               edgecolors='white', linewidths=0.3, label=f'{pop} (n={len(grp)})')
ax.axvline(FAST, color='black', lw=1.2, ls='--', alpha=0.7, label=f'|v| = {FAST} deg/day')
ax.axhline(100, color='grey', lw=0.8, ls=':', alpha=0.6)
# reference: constant absolute offset of 0.2 deg/day → frac = 0.2/|v|
vv = np.logspace(np.log10(speed_mid.min()), np.log10(speed_mid.max()), 100)
ax.plot(vv, (0.2/vv)*100, 'k-', lw=0.8, alpha=0.4, label='constant Δv = 0.2 deg/day')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'midpoint speed $|v_{\rm mid}|$ (deg/day)')
ax.set_ylabel(r'fractional disagreement $|\Delta v|/|v|$  (%)')
ax.legend(fontsize=8, loc='upper right')
ax.set_title('Fractional offset blows up only at low speed\n'
             '(a small-denominator effect, not a measurement failure)', fontsize=10)
ax.grid(alpha=0.2, which='both')

# ── right: fractional distribution for well-conditioned (fast) objects only ────
ax = axes[1]
fast_mask = (speed_mid > FAST).values
ff = frac_dv[fast_mask]
ff_neo = frac_dv[fast_mask & (h.population == 'NEO').values]
bins = np.linspace(0, min(ff.quantile(0.98), 0.5), 25)
ax.hist(ff, bins=bins, color='steelblue', alpha=0.75, edgecolor='white',
        label=f'all |v|>{FAST} (n={fast_mask.sum()})')
if len(ff_neo):
    ax.hist(ff_neo, bins=bins, color=POP_COLORS['NEO'], alpha=0.7, edgecolor='white',
            label=f'NEO only (n={len(ff_neo)})')
ax.axvline(ff.median(), color='black', lw=2, label=f'median = {ff.median()*100:.1f}%')
ax.set_xlabel(r'$|\Delta v|/|v_{\rm mid}|$  (fast objects only)')
ax.set_ylabel('count')
ax.legend(fontsize=9)
ax.set_title(f'Fractional disagreement where the metric is valid\n(|v| > {FAST} deg/day)',
             fontsize=10)
ax.grid(alpha=0.2)

fig.suptitle(f'Speed-normalized agreement — case1 vs benchmark v3, MJD {NIGHT}', fontsize=12)
plt.tight_layout()
plt.savefig('Figures/tracklet_midpoint_normalized_case1.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 9 — Distribution-level median-offset test (identity-free)

Sections 2–8 relied on an (e, H) cKDTree match that we now know pairs *different*
physical objects (see `docs/NOTE_FOR_HYAK_benchmark_objid_mapping.md`).  Until we have
a true BM→S3M identity mapping, we can still ask the underlying question —
**do Sorcha and the benchmark measure the same velocities?** — without pairing
individual objects, by comparing their velocity *distributions* in identical bins.

**The test.**  Bin both catalogs by (population × magnitude bin).  In each cell compute
the median vλ, median vβ, and median |v| separately for benchmark v3 and for the Sorcha
case, then take the **median offset** = median(Sorcha) − median(benchmark).  If the
pipelines/populations agree, per-cell medians land on the 1:1 line and the offsets scatter
around zero.  This is robust to the identity problem: it compares like populations in like
magnitude/velocity regimes, not mismatched individual bodies.

**Null baseline (validity check).**  We also compare the real (e, H)-matched median |Δv|
against the median |Δv| under *randomly shuffled* pairing.  If the real match is no better
than random, the per-object matching carries no identity signal — quantifying exactly how
untrustworthy the Section 6–8 pairing is.

All three Sorcha cases vs benchmark v3, night MJD 61642.

In [ ]:
# ── binned median-velocity comparison (identity-free) ─────────────────────────
MAG_BINS = ['16_18','18_20','mag20','mag21','mag22','mag23','mag24+']
POPS_CMP = ['NEO','MBA','TNO']           # shared, unambiguous populations
MIN_N    = 15                             # need enough objects for a stable median

bench = bv3_raw.copy()
bench['speed'] = np.hypot(bench.vlam, bench.vbeta)

def cell_medians(df):
    df = df.copy()
    df['speed'] = np.hypot(df.vlam, df.vbeta)
    out = {}
    for pop in POPS_CMP:
        for mb in MAG_BINS:
            g = df[(df.population == pop) & (df.mag_bin_label == mb)]
            if len(g) >= MIN_N:
                out[(pop, mb)] = (g.vlam.median(), g.vbeta.median(), g.speed.median(), len(g))
    return out

bench_cells = cell_medians(bench)

rows = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), sharex=True, sharey=True)
for ax, c in zip(axes, ['case1','case2','case3']):
    df = pd.read_parquet(f'docs/sorcha_comparison_{c}.parquet',
                         columns=['night','population','mag_bin_label','vlam','vbeta'])
    sc_cells = cell_medians(df[df.night == NIGHT])
    common = sorted(set(bench_cells) & set(sc_cells))
    for (pop, mb) in common:
        bvl, bvb, bsp, bn = bench_cells[(pop, mb)]
        svl, svb, ssp, sn = sc_cells[(pop, mb)]
        ax.scatter(bsp, ssp, s=30 + 8*np.log10(min(bn, sn)), alpha=0.8,
                   color=POP_COLORS.get(pop, 'grey'), edgecolors='white', linewidths=0.4,
                   label=pop if (pop, mb) == common[[p for p, _ in common].index(pop)] else None)
        rows.append({'case': c, 'population': pop, 'mag_bin': mb,
                     'N_bench': bn, 'N_sorcha': sn,
                     'med|v|_bench': round(bsp, 3), 'med|v|_sorcha': round(ssp, 3),
                     'Δmed_vlam': round(svl - bvl, 3), 'Δmed_vbeta': round(svb - bvb, 3),
                     'Δmed|v|': round(ssp - bsp, 3)})
    lim = 2.2
    ax.plot([0, lim], [0, lim], 'k--', lw=0.9, alpha=0.6)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('benchmark v3  median |v| (deg/day)')
    ax.set_title(f'{CASE_LAB[c]}\n{len(common)} (pop×mag) cells', fontsize=9)
    ax.grid(alpha=0.2); ax.set_aspect('equal')
    # dedup legend
    h, l = ax.get_legend_handles_labels()
    seen = dict(zip(l, h))
    ax.legend(seen.values(), seen.keys(), fontsize=8, loc='upper left')
axes[0].set_ylabel('Sorcha  median |v| (deg/day)')
fig.suptitle(f'Per-bin median speed: Sorcha vs benchmark v3  (on 1:1 line = distributions agree)  |  MJD {NIGHT}',
             fontsize=11)
plt.tight_layout()
plt.savefig('Figures/median_offset_binned.png', dpi=150, bbox_inches='tight')
plt.show()

offs = pd.DataFrame(rows)
print('=== per-(case, population) summary of median velocity offset (Sorcha − benchmark) ===')
summ = (offs.groupby(['case','population'])
            .agg(n_cells=('Δmed|v|','size'),
                 med_Δvlam=('Δmed_vlam','median'),
                 med_Δvbeta=('Δmed_vbeta','median'),
                 med_Δspeed=('Δmed|v|','median'),
                 rms_Δspeed=('Δmed|v|', lambda x: round(np.sqrt(np.mean(x**2)),3)))
            .round(3))
display(summ)

In [ ]:
# ── null baseline: is the (e,H) match any better than random pairing? ─────────
# For case1's matched pairs, compare real median |Δv| against the median |Δv|
# obtained by randomly shuffling which benchmark object each Sorcha object is paired to.
rng = np.random.default_rng(0)
hh = matched['case1']                       # the (e,H)-matched pairs from Section 5
real_dv = hh.dv.values
real_med = np.median(real_dv)

bench_v = bv3[['vlam','vbeta']].values       # pool of benchmark velocities to draw from
sc_v    = hh[['vlam','vbeta']].values
N = len(hh)
shuf_meds = []
for _ in range(500):
    draw = bench_v[rng.integers(0, len(bench_v), size=N)]
    d = np.hypot(sc_v[:,0]-draw[:,0], sc_v[:,1]-draw[:,1])
    shuf_meds.append(np.median(d))
shuf_meds = np.array(shuf_meds)

print('=== Null-baseline test (case1) ===')
print(f'  real (e,H)-matched median |Δv|:      {real_med:.4f} deg/day')
print(f'  random-pairing median |Δv|:          {shuf_meds.mean():.4f} ± {shuf_meds.std():.4f} deg/day')
print(f'  improvement over random:             {100*(1-real_med/shuf_meds.mean()):.1f}%')
z = (real_med - shuf_meds.mean()) / shuf_meds.std()
print(f'  z vs null:                           {z:.1f}')
print('  → if improvement ≈ 0%, the (e,H) match carries no identity signal for this population mix.')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(shuf_meds, bins=30, color='grey', alpha=0.7, edgecolor='white',
        label=f'random pairing (n=500)')
ax.axvline(real_med, color='tab:red', lw=2.5, label=f'real (e,H) match = {real_med:.3f}')
ax.axvline(shuf_meds.mean(), color='black', lw=1.5, ls='--',
           label=f'random mean = {shuf_meds.mean():.3f}')
ax.set_xlabel('median |Δv| (deg/day)')
ax.set_ylabel('count')
ax.legend(fontsize=9)
ax.set_title('Does the (e,H) match beat random pairing?\n'
             'gap between red and dashed = real identity signal', fontsize=10)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('Figures/median_offset_nullbaseline.png', dpi=150, bbox_inches='tight')
plt.show()

### What the median-offset test shows

**Binned distributions (Sorcha − benchmark median velocity), consistent across all 3 cases:**

| population | Δmed \|v\| | Δmed vλ | Δmed vβ | reading |
|---|---:|---:|---:|---|
| TNO | +0.005 | −0.01 | 0.00 | distributions essentially identical |
| MBA | −0.04 | −0.17 | 0.00 | small: Sorcha MBAs slightly slower in vλ |
| NEO | −0.25 | **−0.53** | +0.04 | large: Sorcha NEO velocity distribution is shifted |

- **TNO and MBA velocity distributions agree** between all-sky benchmark and footprint-limited
  Sorcha — the slow populations look the same in both.
- **NEO distributions differ substantially** (median vλ lower by 0.53 deg/day in Sorcha).  This
  is a **selection effect, not a measurement error**: the benchmark is all-sky and includes fast,
  high-elongation NEOs, while Sorcha only keeps NEOs that were actually detected in real visits —
  and fast movers are preferentially lost (trailing, brief visibility, motion cuts).  So Sorcha's
  surviving NEO sample skews slower.  The offset being *identical* across case1/2/3 confirms it's
  selection/geometry, not linking.

**Null baseline — how much identity does the (e,H) match actually carry?**

- real (e,H)-matched median |Δv| = **0.191** deg/day
- random-pairing median |Δv| = **0.253 ± 0.009** deg/day  → real is **24% better than random (z = −6.7)**

So the (e,H) match is *weakly informative* — objects with similar (e, H) do have somewhat more
similar velocities than random pairs (eccentricity and brightness correlate with orbital rate),
and the improvement is statistically real.  But it is **nowhere near identity-level**: ~¾ of the
per-object residual is indistinguishable from random pairing.  This is the quantitative statement
of why the per-object plots (Sections 6–8) can't be read as same-object agreement, while the
**distribution-level test above is valid** and gives the trustworthy answer:
*slow populations agree; the NEO difference is a survey selection effect.*